In [1]:
import pandas as pd

# 1. Load the raw dataset
print("Loading raw data...")
df = pd.read_csv('../data/raw/weather_prediction_dataset.csv')

# 2. Isolate DATE and Basel columns to keep the dataset clean for inspection
basel_cols = [col for col in df.columns if 'BASEL' in col]
df_basel = df[['DATE'] + basel_cols].copy()

# 3. Create the continuous target (Tomorrow's minimum temperature)
df_basel['target_future_min_temp'] = df_basel['BASEL_temp_min'].shift(-1)

# 4. Apply the threshold rule (<= 3.0°C) to generate the binary label
# This follows the physical/threshold framing for cold-injury risk
threshold = 3.0
df_basel['frost_label'] = (df_basel['target_future_min_temp'] <= threshold).astype(int)

# 5. Drop the final row (which has no 'tomorrow' to predict)
df_basel = df_basel.dropna(subset=['target_future_min_temp'])

# 6. Save the labeled dataset for inspection and pipeline use
output_path = '../data/interim/basel_labeled.csv'
df_basel.to_csv(output_path, index=False)

print(f"Done! Labeled dataset saved to {output_path}")
print(f"Total rows: {len(df_basel)}")
print(f"Frost events (Label 1): {df_basel['frost_label'].sum()}")
print("\nFirst 5 rows of your target variables:")
display(df_basel[['DATE', 'BASEL_temp_min', 'target_future_min_temp', 'frost_label']].head())

Loading raw data...
Done! Labeled dataset saved to ../data/interim/basel_labeled.csv
Total rows: 3653
Frost events (Label 1): 1083

First 5 rows of your target variables:


,DATE,BASEL_temp_min,target_future_min_temp,frost_label
0,20000101,1.6,2.7,1
1,20000102,2.7,0.1,1
2,20000103,0.1,0.5,1
3,20000104,0.5,3.8,0
4,20000105,3.8,1.9,1


In [2]:
# Load the newly labeled dataset
df_basel = pd.read_csv('../data/interim/basel_labeled.csv')

# Group by our new label to compare average weather conditions today 
# that lead to frost tomorrow.
inspection_cols = [
    'BASEL_cloud_cover', 
    'BASEL_wind_speed', 
    'BASEL_humidity', 
    'BASEL_temp_min'
]

print("--- Average Conditions: Safe Days (0) vs. Cold-Risk Days (1) ---")
display(df_basel.groupby('frost_label')[inspection_cols].mean().round(2))

# Let's also look at the extremes: What did the absolute coldest day look like?
worst_frost_day = df_basel.loc[df_basel['target_future_min_temp'].idxmin()]
print("\n--- The Coldest Event in the Dataset ---")
print(f"Date: {worst_frost_day['DATE']}")
print(f"Tomorrow's Min Temp: {worst_frost_day['target_future_min_temp']}°C")
print(f"Today's Cloud Cover: {worst_frost_day['BASEL_cloud_cover']}")
print(f"Today's Wind Speed: {worst_frost_day['BASEL_wind_speed']}")
print(f"Today's Humidity: {worst_frost_day['BASEL_humidity']}")

--- Average Conditions: Safe Days (0) vs. Cold-Risk Days (1) ---


KeyError: "Columns not found: 'BASEL_wind_speed'"